# Смешанные модели для бинарных данных (GLMM)

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/10_mixed_models_logistic_ru.ipynb)
*English version: [10_mixed_models_logistic.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/10_mixed_models_logistic.ipynb)*

Эта рабочая тетрадь — приложение к занятию 10 «Бинарные исходы и обобщённые линейкие модели» и странице «Классические смешанные модели и байесовская иерархия». Мы разберём продольные бинарные данные из клинического испытания лечения депрессии: от обычной логистической регрессии к смешанной модели со случайным интерсептом пациента, сначала в statsmodels, затем в Bambi, и сравним модели по PSIS-LOO. В конце — пример о неоднородных группах без повторений: голосование на выборах ANES 2016.


## Продольное исследование лечения депрессии

Table 9.1 refers to a longitudinal study comparing a new drug with a standard drug for treating subjects suffering mental depression. Subjects were classified into two groups according to whether the initial severity of depression was mild or severe. In each group, subjects were randomly assigned to one of the two drugs. Following 1 week, 2 weeks, and 4 weeks of treatment, each subject's extent of suffering from mental depression was classified as normal or abnormal.
- $y_t$ = 1 for normal and 0 for abnormal at time $t$.

Пусть $s$ обозначает исходную тяжесть состояния: $s = 1$ — тяжёлая, $s = 0$ — лёгкая. Пусть $d$ обозначает препарат: $d = 1$ — новый, $d = 0$ — стандартный. Пусть $t$ — момент измерения. Когда шкала времени отражает накопленную дозу препарата, на логит-шкале эффект часто приблизительно линеен по логарифму времени. Используются баллы $(0, 1, 2)$ — логарифмы номеров недель $(1, 2, 4)$ по основанию 2. С самими номерами недель получаются похожие содержательные результаты.

(Источник данных: Agresti, *Categorical Data Analysis*, таблица 9.1. Файл `Depression.dat` входит в репозиторий курса; оригинал распространялся по адресу users.stat.ufl.edu/~aa/cat/data/Depression.dat.)


In [ ]:
import os, urllib.request
import pandas as pd
import arviz as az
import seaborn as sns
import bambi as bmb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

az.style.use("arviz-darkgrid")

RANDOM_SEED = 2026

# Данные — из репозитория курса (оригинальный ./file_examples/ больше не существует)
DEPR_PATH = "data/Depression.dat"
os.makedirs("data", exist_ok=True)
if not os.path.exists(DEPR_PATH):
    url = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/Depression.dat"
    urllib.request.urlretrieve(url, DEPR_PATH)


In [ ]:
data_depr = pd.read_csv(DEPR_PATH, sep=r"\s+")
data_depr.head()


In [ ]:
agg_data = data_depr.groupby(['drug', 'severity', 'time'])['outcome'].mean().reset_index()
agg_data


In [ ]:
sns.catplot(x="time", y="outcome", hue="drug", col='severity', data=agg_data, kind="bar",
            palette="rocket", legend_out=False)
plt.show()


In [ ]:
basic = smf.logit(formula = 'outcome ~ drug', data = data_depr).fit()
print(basic.summary())


In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (10, 3))
axs[0].scatter(range(len(data_depr)), basic.resid_generalized)
axs[1].scatter(basic.fittedvalues, basic.resid_generalized)
plt.show()


У модели только с препаратом остатки структурированы: модель не учитывает ни тяжесть состояния, ни время. Добавим в модель все имеющиеся предикторы и взаимодействие препарата со временем.


In [ ]:
basic = smf.logit(formula = 'outcome ~ drug + time + severity + drug:time', data = data_depr).fit()
print(basic.summary())


In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (10, 3))
axs[0].scatter(range(len(data_depr)), basic.resid_generalized)
axs[1].scatter(basic.fittedvalues, basic.resid_generalized)
plt.show()


Обратите внимание: в полной модели главный эффект препарата почти нулевой ($\approx -0.06$), а взаимодействие препарата со временем большое ($\approx +1.02$). Это содержательный результат: на старте препараты не различаются, различие нарастает с накопленной дозой. Главное влияние без взаимодействия здесь ничего не значит.

У бинарных данных остатки всегда образуют две полосы (возможные значения $y - p$ ограничены), поэтому сам по себе такой график малоинформативен; важно, что плоская модель в принципе не учитывает повторяемость наблюдений одного пациента — 1020 строк ведут себя как 1020 независимых людей, и стандартные ошибки занижены.


In [ ]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM


In [ ]:
random = {'a': '0 + C(case)', }
mixed = BinomialBayesMixedGLM.from_formula(
    'outcome ~ drug + time + severity + drug:time', random, data_depr)
result = mixed.fit_vb()


In [ ]:
print(result.summary())


In [ ]:
result.random_effects().head()


Если посмотреть на точечные оценки, то кажется, что всё одинаковое. Проблема в том, что не оценить качество, и выводы сложно делать какие-то, особенно количественные. Насколько? Какой размер эффекта? Какая неопределённость? Вариационное приближение в statsmodels даёт приближённые средние и стандартные отклонения, но не даёт ни сэмплов апостериорного распределения, ни оценок его качества, ни распределения для дисперсии случайных эффектов — только одно число.


## Байесовские смешанные модели в Bambi

Теперь те же модели в Bambi: сначала плоская модель только с препаратом, затем полная плоская модель, затем полная модель со случайным интерсептом пациента `(1 | case)`. Сравним их по PSIS-LOO (в прежней версии тетради использовался WAIC; в актуальном ArviZ остался критерий LOO).


In [ ]:
model1 = bmb.Model("outcome ~ drug", data_depr, family="bernoulli")
model1


In [ ]:
idata1 = model1.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                     idata_kwargs={"log_likelihood": True})


In [ ]:
print(az.loo(idata1))
az.summary(idata1)


In [ ]:
model2 = bmb.Model("outcome ~ drug + time + severity + drug:time", data_depr, family="bernoulli")
idata2 = model2.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    idata_kwargs={"log_likelihood": True})
print(az.loo(idata2))
az.summary(idata2)


In [ ]:
model3 = bmb.Model("outcome ~ drug + time + severity + drug:time + (1 | case)",
                    data_depr, family="bernoulli")
idata3 = model3.fit(draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED,
                    target_accept=0.95,
                    idata_kwargs={"log_likelihood": True})
print(az.loo(idata3))
# Полная сводка по 340 пациентским эффектам неинформативна — смотрим популяционные параметры
az.summary(idata3, var_names=["Intercept", "drug", "time", "severity", "drug:time", "1|case_sigma"])


In [ ]:
model3


In [ ]:
models_dict = {
    "model1: drug": idata1,
    "model2: полная плоская": idata2,
    "model3: + (1|case)": idata3,
}
df_compare = az.compare(models_dict)
df_compare


In [ ]:
az.plot_compare(df_compare)
plt.show()


Как читать этот результат. Полная плоская модель и смешанная модель практически неразличимы по предсказательной точности (разница меньше одной единицы ELPD). Причина понятна: на каждого пациента приходится всего три бинарных наблюдения, и данные не позволяют отличить «пациенты не различаются» от «различия есть, но небольшие». Апостериорное распределение `1|case_sigma` сосредоточено около нуля (среднее $\approx 0.3$), то есть частичное объединение вырождается в полное. Это не значит, что иерархические модели бесполезны: на данных `sleepstudy` из тетради занятия 9, где на каждого испытуемого приходится десять непрерывных наблюдений, иерархическая структура выигрывала уверенно. Иерархия окупается тогда, когда кластеры информативны.

Отдельно стоит сказать про графики для содержательного вывода: коэффициенты логистической регрессии живут на шкале лог-шансов, и их нужно переводить в вероятности при явно названных условиях. Например, для типичного пациента к четвёртой неделе новый препарат повышает вероятность нормальной классификации примерно на $+0.23$ для исходно лёгкой формы и на $+0.43$ для тяжёлой (94% интервалы $[+0.17, +0.29]$ и $[+0.34, +0.51]$). Соответствующие расчёты есть на странице «Классические смешанные модели и байесовская иерархия».


## Пример без повторений: голосование на выборах ANES 2016

Не все бинарные данные продольные. Смешанно-модельный аппарат полезен и для неоднородных групп в одном срезе. В данных ANES 2016 моделируем голосование за Клинтон: зависимая переменная `vote['clinton']` означает, что успехом считается голос за Клинтона. Формула `vote['clinton'] ~ party_id + party_id:age` даёт каждой партии собственный наклон возраста: взаимодействие `party_id:age` разворачивается в три коэффициента (по одному на демократов, независимых и республиканцев), поэтому отдельный главный эффект возраста не нужен и не пропущен.


In [ ]:
# Подмножество ANES 2016 (голоса clinton/trump) — из репозитория курса
ANES_PATH = "data/anes_clinton.csv"
if not os.path.exists(ANES_PATH):
    url_a = "https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/anes_clinton.csv"
    urllib.request.urlretrieve(url_a, ANES_PATH)
data = pd.read_csv(ANES_PATH)
data.head()


In [ ]:
clinton_data = data.loc[data["vote"].isin(["clinton", "trump"]), :]
clinton_data.head()


In [ ]:
clinton_model = bmb.Model("vote['clinton'] ~ party_id + party_id:age", clinton_data, family="bernoulli")
clinton_fitted = clinton_model.fit(
    draws=1000, tune=1000, chains=4, random_seed=RANDOM_SEED, target_accept=0.85,
    idata_kwargs={"log_likelihood": True}
)


In [ ]:
az.summary(clinton_fitted)


In [ ]:
# Наклоны возраста по партиям на шкале лог-шансов
post = clinton_fitted.posterior.to_dataset()
party_levels = ["democrat", "independent", "republican"]
slopes = {p: post["party_id:age"].sel(**{"party_id:age_dim": p}).values.flatten()
          for p in party_levels}

fig, ax = plt.subplots(figsize=(7, 3.8))
for i, p in enumerate(party_levels):
    ax.hist(slopes[p], bins=60, alpha=0.55, density=True, label=p, color=f"C{i}")
ax.set_xlabel("наклон возраста (лог-шансы за год)")
ax.legend()
plt.show()

# Те же эффекты на шкале вероятностей: изменение вероятности за десять лет возраста
# при собственной базовой вероятности каждой партии
for p in party_levels:
    eta = post["Intercept"].values.flatten()
    if p != "democrat":   # democrat — референтный уровень, входит в интерсепт
        eta = eta + post["party_id"].sel(**{"party_id_dim": p}).values.flatten()
    p_hat = 1 / (1 + np.exp(-eta))
    delta10 = slopes[p] * 10 * p_hat * (1 - p_hat)
    lo, hi = az.hdi(delta10, prob=0.94)
    print(f"{p:12s}: изменение P(голос за Клинтон) за 10 лет возраста = {delta10.mean():+.3f}, "
          f"94% HDI [{lo:+.3f}, {hi:+.3f}]")


Наклоны у партий почти не пересекаются: с возрастом вероятность голоса за Клинтона заметно падает у республиканцев и независимых, а у демократов близка к нулю. Один «средний» эффект возраста скрыл бы всю картину — взаимодействия делают неоднородность видимой.

## Домашнее задание

Примените разобранный сегодня подход к данным о просоциальных шимпанзе (капстоун занятия 10, файл `data/chimpanzees.csv`):

- постройте плоскую логистическую модель `pulled_left ~ prosoc_left * condition`, игнорируя акторов и блоки;
- постройте смешанную модель `pulled_left ~ prosoc_left * condition + (1 | actor) + (1 | block)`;
- сравните модели по PSIS-LOO и отчитайтесь о популяционном эффекте просоциальной опции с неопределённостью;
- одно предложение: что даёт включение `(1 | block)` и поддерживает ли его таблица LOO.
